# 00-05 - Tables with pandas

In 00-04 we kept the kinds of place in one list and their counts in another, and we had to keep them lined up ourselves. Real data does not arrive like that. It arrives as a table, with rows and columns and a name at the top of each column, and it has thousands of rows rather than six.

The library for tables in Python is called **pandas**, and it is the single most useful thing in this series. We meet it on 92 places to eat, which is small enough to look at whole, and then on 7893 buildings, which is not.

In [1]:
import pandas as pd

## Reading a table

Our data is in a `.csv` file, which is plain text with commas between the values and a row of column names at the top. `read_csv` turns one into a table:

In [2]:
food = pd.read_csv("data/food_table.csv")
food.head()

,name,amenity,lat,lon
0,Metro North,restaurant,40.33471,-74.65398
1,Roots Ocean Prime,restaurant,40.34333,-74.65944
2,Starbucks,cafe,40.34993,-74.65946
3,Tacoria,fast_food,40.35003,-74.65912
4,Maruichi Japanese Food & Deli,restaurant,40.35026,-74.65827


The first five rows. Four columns, `name`, `amenity`, `lat` and `lon`, and a bold column of numbers down the left which is the **index**, i.e. the label of each row.

`head` shows five rows by default and takes a number if you want a different amount. Its opposite is `tail`, which shows the last rows, and is the fastest way to notice that a file has a stray blank line at the bottom.

Note the file path. `"data/food_table.csv"` means "starting from the folder this notebook is in, go into `data`". Getting that wrong is the `FileNotFoundError` from 00-07.

## How big is it, and what is in it

Three things are worth asking of any table before you do anything else:

In [3]:
print(food.shape)
print(list(food.columns))
print(food.dtypes)

(92, 4)
['name', 'amenity', 'lat', 'lon']
name           str
amenity        str
lat        float64
lon        float64
dtype: object


`(92, 4)` means 92 rows and 4 columns, always in that order.

The column names come next, and it is worth reading them rather than assuming. Capital letters matter, and asking for a column that does not exist by exactly that spelling is the `KeyError` we meet in 00-07.

The last block is the **dtype** of each column, i.e. what kind of value it holds. `object` means text, and `float64` means a number with a decimal point. Note that `lat` and `lon` are numbers, which is what lets us do arithmetic on them in 00-06. If they had arrived as text, everything downstream would misbehave quietly.

Note that `shape` and `columns` and `dtypes` have no brackets after them. They are properties of the table rather than things it does, which is the distinction from 00-03.

## One column

A single column is reached by name, in square brackets:

In [4]:
food["amenity"].head()

0    restaurant
1    restaurant
2          cafe
3     fast_food
4    restaurant
Name: amenity, dtype: str

One column of values, with the index still down the left.

A single column is called a **Series**, and a whole table is a **DataFrame**. The difference matters occasionally, and the practical version is that a Series is one column and a DataFrame is a rectangle.

## What is actually in a column

Here is the method that earns its keep more than any other. `value_counts` tells you what values a column contains and how often each appears:

In [5]:
food["amenity"].value_counts()

amenity
restaurant    52
fast_food     16
cafe          15
ice_cream      6
pub            2
bar            1
Name: count, dtype: int64

Fifty-two restaurants, sixteen fast food places, fifteen cafes, six ice cream shops, two pubs and one bar.

That is the same data we typed by hand into two lists in 00-04, except that we did not have to know it in advance and it cannot fall out of step with itself. This is what a table buys you.

Note that the six numbers add to 92, the number of rows. When a breakdown adds to the total you already know, nothing was dropped and nothing was double counted, and that is a check worth doing.

Note also that this vocabulary is not ours. `fast_food` and `ice_cream` are OpenStreetMap's words, chosen by the people who built the map, and we have to use them exactly. Asking for `pizza` or `takeaway` returns nothing at all.

## Choosing rows

To keep only some rows, first write a comparison. It gives back one True or False per row:

In [6]:
is_cafe = food["amenity"] == "cafe"
print(is_cafe.head())
print(f"{is_cafe.sum()} True values out of {len(is_cafe)}")

0    False
1    False
2     True
3    False
4    False
Name: amenity, dtype: bool
15 True values out of 92


A column of True and False, and 15 of them are True.

`sum` counting Trues is not a trick. Python treats True as one and False as zero, so adding up a column of them counts how many were true. Almost every count in this course is produced this way.

Now, putting that column of True and False inside the table's square brackets keeps the rows where it said True:

In [7]:
cafes = food[food["amenity"] == "cafe"]
cafes.head()

,name,amenity,lat,lon
2,Starbucks,cafe,40.34993,-74.65946
17,illy Coffee At Earth's End Princeton,cafe,40.35118,-74.65848
18,Ellinikon,cafe,40.35130,-74.65481
24,Fresh Ó Tea,cafe,40.35220,-74.65175
29,Small World Coffee,cafe,40.35238,-74.65120


Fifteen rows, all cafes, with their original index numbers preserved so you can see which rows of the original they were.

Read that line again, because it is the most important pattern in this notebook: `food[food["amenity"] == "cafe"]`. The inner part makes a column of True and False, and the outer square brackets use it to choose rows. The table appears twice in one line, which looks odd until you see the two jobs it is doing.

## A table you cannot look at

Everything so far would have been possible by hand with 92 rows. Now a table that would not be:

In [8]:
buildings = pd.read_csv("data/buildings_table.csv")

print(buildings.shape)
buildings.head()

(7893, 2)


,name,building
0,NaN,house
1,NaN,house
2,NaN,house
3,NaN,yes
4,NaN,house


7893 rows and two columns, `name` and `building`.

Note the word `NaN` in the `name` column. It means **not a number**, which is pandas's way of saying there is no value here. It is not zero and not an empty string, it is an absence, and it is about to be the most interesting thing in this notebook.

In [9]:
buildings["building"].value_counts().head(8)

building
house          4135
yes            2958
university      177
retail          126
garage          107
residential      93
shed             69
dormitory        68
Name: count, dtype: int64

`house` 4135, `yes` 2958, then `university`, `retail`, `garage` and a long tail.

Stop at `yes`. That is not a kind of building, it is somebody drawing an outline on the map and saying "there is a building here" without saying what sort. Nearly three thousand of them, and together with `house` they are 89.9 percent of every building in the dataset.

So a question like "how many shops are there in Princeton" cannot be answered from this column with any confidence, because the overwhelming majority of buildings simply are not classified. The column is not wrong; it is a record of what volunteers chose to write down.

## Counting what is missing

The `name` column is even starker. `isna` gives True where a value is absent:

In [10]:
missing = buildings["name"].isna().sum()

print(f"{missing:,} of {len(buildings):,} buildings have no name")
print(f"that is {missing / len(buildings) * 100:.1f} percent")

7,505 of 7,893 buildings have no name
that is 95.1 percent


7,505 of 7,893, i.e. 95.1 percent, have no name at all.

This is the sentence to take away from the whole foundations series. **A dataset is a record of what somebody bothered to write down, not an inventory of the world.** Princeton has not got 388 named buildings; it has 388 buildings that somebody thought worth naming on a volunteer map, and the rest are outlines.

Note how cheaply we learned that: one method, one line. Running `isna().sum()` on a column you are about to rely on takes five seconds and occasionally saves an entire analysis.

## Numbers in a column

For a column of numbers, `describe` gives a summary in one call:

In [11]:
food[["lat", "lon"]].describe()

,lat,lon
count,92.000000,92.000000
mean,40.352522,-74.657340
std,0.004923,0.004562
min,40.334710,-74.664780
25%,40.350417,-74.660742
50%,40.351010,-74.659545
75%,40.352305,-74.652140
max,40.364650,-74.645060


Count, mean, standard deviation, minimum, maximum and the quartiles, for both coordinate columns.

The two rows worth reading are `min` and `max`. The latitudes run from about 40.335 to 40.365 and the longitudes from about -74.665 to -74.645, i.e. a box roughly three hundredths of a degree tall and two hundredths wide. In 00-06 we turn that box into a picture.

Note the double square brackets in `food[["lat", "lon"]]`. A single pair with one name gives a column; a double pair with a list of names gives a smaller table. Both are useful and the difference catches everybody once.

## Check your understanding

1. `food.shape` gives `(92, 4)`. Which number is the rows?
2. What does `len(food[food["amenity"] == "pizza"])` return, and why is it not an error?
3. What is the difference between `buildings["building"]` and `buildings[["building"]]`?
4. `buildings["name"].notna().sum()` returns 388. Say in one sentence what that number means about Princeton, being careful about what it does not mean.

## Where we are

You can read a table, see how big it is, look at what a column contains, choose rows with a condition, count with a sum, and find what is missing. That is genuinely most of what data work consists of.

In 00-06 we take the two coordinate columns and draw them, and discover that a map is not as mysterious as it looks.

## Further resources

Nothing in this course requires anything below.

The pandas "10 minutes to pandas" guide covers this material at a brisker pace and goes further: https://pandas.pydata.org/docs/user_guide/10min.html. The indexing guide is the reference for choosing rows and columns, which is the part with the most corners: https://pandas.pydata.org/docs/user_guide/indexing.html.

The tables used here are an OpenStreetMap extract of Princeton, snapshot 2026-08-29, licensed ODbL 1.0, copyright OpenStreetMap contributors: https://www.openstreetmap.org/copyright.